In [ ]:
# === Configuration ===
EPOCHS = 10              # MINGUS paper recipe (10 epochs per phase)
BPTT = 35                # sequence length (default in train.py)
SMOKE_MAX_EPOCH = None   # set to 1-2 for fast pipeline test on Colab; None = full recipe

# === Paths ===
REPO_ROOT = "/content/repo"
GIT_URL = "https://github.com/kudrmax/MINGUS.git"
GIT_BRANCH = "master"    # NOTE: switch to feat/colab-training while iterating; master after merge

DIPLOMA_REPO_ROOT = "/content/diploma2"
DIPLOMA_GIT_URL = "https://github.com/kudrmax/jazz-generation-research.git"
DIPLOMA_BRANCH = "master"  # same note as above

DRIVE_ROOT = "/content/drive/MyDrive/mingus-training"
XML_DIR = "A_preprocessData/data/xml"
DATA_JSON_PATH = "A_preprocessData/data/DATA.json"

# Conditioning features (paper §3.2 optimal values)
COND_TYPE_PITCH = "D-C-B-BE-O"
COND_TYPE_DURATION = "B-BE-O"

# Distinct run dir per (conditioning, epochs) so multiple ablations coexist
# on Drive without resume colliding across them.
RUN_NAME = f"result-pitch-{COND_TYPE_PITCH}-dur-{COND_TYPE_DURATION}-ep{EPOCHS}"
RESULT_ROOT = f"{DRIVE_ROOT}/{RUN_NAME}"

import threading, time, os, re, subprocess
from datetime import datetime


def run(cmd, **kw):
    """Popen + line-by-line stdout proxy. Identical pattern to CMT colab notebook
    — Colab's IOPub buffers child stdout indirectly through the parent until
    the child exits, which makes long-running commands look frozen. Re-printing
    each line through parent keeps the cell alive in the UI."""
    print(f">>> {cmd}", flush=True)
    env = kw.pop("env", None) or os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd, shell=isinstance(cmd, str), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True, **kw,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return proc


def cache_restore(name, dest_path):
    """Restore <DRIVE_ROOT>/cache/<name>.tar.gz to dest_path. Returns True if restored."""
    tar_path = f"{DRIVE_ROOT}/cache/{name}.tar.gz"
    if os.path.exists(dest_path):
        return False
    if not os.path.exists(tar_path):
        return False
    print(f"==> Restoring {name} from Drive cache...", flush=True)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    run(["tar", "-xzf", tar_path, "-C", os.path.dirname(dest_path)])
    return True


def cache_save(name, src_path):
    """Cache src_path to <DRIVE_ROOT>/cache/<name>.tar.gz. No-op if missing or already cached."""
    tar_path = f"{DRIVE_ROOT}/cache/{name}.tar.gz"
    if not os.path.exists(src_path):
        return False
    if os.path.exists(tar_path):
        return False
    print(f"==> Caching {name} to Drive...", flush=True)
    os.makedirs(os.path.dirname(tar_path), exist_ok=True)
    run(["tar", "-czf", tar_path, "-C", os.path.dirname(src_path), os.path.basename(src_path)])
    return True


# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/cache", exist_ok=True)

# 2. Fresh clone MINGUS fork
if os.path.isdir(REPO_ROOT):
    backup = f"{REPO_ROOT}.old.{int(time.time())}"
    os.rename(REPO_ROOT, backup)
    print(f"==> Moved old repo to {backup}", flush=True)
run(["git", "clone", "-b", GIT_BRANCH, GIT_URL, REPO_ROOT])

# 3. Clone diploma2 main repo for split.json (shallow, no submodules needed)
if os.path.isdir(DIPLOMA_REPO_ROOT):
    os.rename(DIPLOMA_REPO_ROOT, f"{DIPLOMA_REPO_ROOT}.old.{int(time.time())}")
run(["git", "clone", "--depth", "1", "-b", DIPLOMA_BRANCH, DIPLOMA_GIT_URL, DIPLOMA_REPO_ROOT])
os.chdir(REPO_ROOT)

# 4. Install MINGUS deps
run(["pip", "install", "-q", "-r", "requirements-py312.txt"])

# 5. Sanity: split.json present
SPLIT_JSON = f"{DIPLOMA_REPO_ROOT}/pipelines/training-pipeline/wjazzd_split.json"
assert os.path.exists(SPLIT_JSON), f"split.json missing at {SPLIT_JSON}"

# 6. Sanity-check: xml/ committed to MINGUS fork (430 → 422 files generated locally
#    one-time via Task 2 Step 2.7 and committed; should be present after git clone).
xml_full = os.path.join(REPO_ROOT, XML_DIR)
if os.path.isdir(xml_full):
    n_xml = sum(
        1 for f in os.listdir(xml_full)
        if f.endswith(".xml") and len(f) >= 4 and f[:3].isdigit() and f[3] == "_"
    )
else:
    n_xml = 0
assert n_xml >= 420, (
    f"xml/ missing or incomplete ({n_xml} new-format files); "
    f"run extract_wjazzd_csv + wjazzDB_csv_to_xml locally and commit per Task 2 Step 2.7"
)
print(f"==> Input xml OK: {n_xml} new-format files in {xml_full}", flush=True)

# 7. Restore-or-preprocess DATA.json
data_json_full = os.path.join(REPO_ROOT, DATA_JSON_PATH)
cache_restore("mingus_data_json", data_json_full)
if not os.path.exists(data_json_full):
    print("==> Running data_preprocessing.py...", flush=True)
    run(["python", "-m", "A_preprocessData.data_preprocessing",
         "--format", "xml", "--split-json", SPLIT_JSON])
    assert os.path.exists(data_json_full)
cache_save("mingus_data_json", data_json_full)

# 8. ETA monitor — parse train.log emitted per epoch by train.py
def monitor_loop():
    log_re = re.compile(r"(pitch|duration) end of epoch\s+(\d+)\s+\|\s+time:\s+([\d\.]+)s")
    while True:
        time.sleep(60)
        try:
            log_path = f"{RESULT_ROOT}/train.log"
            if not os.path.exists(log_path):
                continue
            text = open(log_path).read()
            matches = log_re.findall(text)
            if not matches:
                continue
            last_phase, last_epoch_str, _ = matches[-1]
            last_epoch = int(last_epoch_str)
            avg_time = sum(float(t) for _, _, t in matches) / len(matches)
            # Total = 2 * EPOCHS (pitch + duration). Done = sum of done epochs across both phases.
            total_epochs = 2 * EPOCHS
            done_epochs = len(matches)
            eta_min = (total_epochs - done_epochs) * avg_time / 60
            print(f"[ETA] {last_phase} epoch {last_epoch}/{EPOCHS}, "
                  f"{done_epochs}/{total_epochs} total epochs done, "
                  f"avg {avg_time:.0f}s/epoch, ETA {eta_min:.1f} min", flush=True)
        except Exception:
            pass

threading.Thread(target=monitor_loop, daemon=True).start()

# 9. Train (resumes automatically from <work_dir>)
os.makedirs(RESULT_ROOT, exist_ok=True)
train_cmd = ["python", "-m", "B_train.train",
             "--EPOCHS", str(EPOCHS),
             "--BPTT", str(BPTT),
             "--COND_TYPE_PITCH", COND_TYPE_PITCH,
             "--COND_TYPE_DURATION", COND_TYPE_DURATION,
             "--work-dir", RESULT_ROOT]
if SMOKE_MAX_EPOCH:
    train_cmd[train_cmd.index("--EPOCHS") + 1] = str(SMOKE_MAX_EPOCH)
run(train_cmd)

# 10. Confirm final artefacts
run(["ls", "-la", RESULT_ROOT])
run(["ls", "-la", f"{RESULT_ROOT}/pitchModel"])
run(["ls", "-la", f"{RESULT_ROOT}/durationModel"])
assert os.path.exists(f"{RESULT_ROOT}/pitch.done")
assert os.path.exists(f"{RESULT_ROOT}/duration.done")
assert os.path.exists(f"{RESULT_ROOT}/pitch_best.pt")
assert os.path.exists(f"{RESULT_ROOT}/duration_best.pt")
final_epochs = SMOKE_MAX_EPOCH if SMOKE_MAX_EPOCH else EPOCHS
assert os.path.exists(f"{RESULT_ROOT}/pitchModel/MINGUS COND {COND_TYPE_PITCH} Epochs {final_epochs}.pt")
assert os.path.exists(f"{RESULT_ROOT}/durationModel/MINGUS COND {COND_TYPE_DURATION} Epochs {final_epochs}.pt")
print("==> All MINGUS training artefacts present.", flush=True)
